# Trees & Heaps — Inventing Fast Lookup

You keep a collection of numbers. Items are **inserted**, **deleted**, and
**searched** constantly. What is the best data structure?

| Structure | insert | find | delete |
|-----------|--------|------|--------|
| plain list | O(1) append | O(n) scan | O(n) find + O(n) shift |
| sorted list | O(n) shift | O(log n) binary search | O(n) shift |
| dictionary | O(1) | O(1) | O(1) — but heavy memory, no ordering |

Each structure is fast at some operations and slow at others.
In this chapter you will invent a structure that does **all three in O(log n)**
while keeping the data ordered — the **binary search tree** — and then a
specialized cousin that powers every task scheduler: the **heap**.

In [ ]:
import random

---
## Part 1 — Warm-Up: Trees That Compute

A **tree** is nodes connected downward: each node holds a value and has up to
two children, `left` and `right`.

Here is the arithmetic expression `2 * 3 + 4` drawn as a tree:

```
        +
       / \
      *   4
     / \
    2   3
```

Leaves are numbers; internal nodes are operators. This is exactly how compilers
see your code (Python, SQL, C++ — everything is parsed into trees).

The `Node` class and a tree printer are provided — run this cell as-is.

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def print_tree(node, indent=0):
    """Prints the tree sideways: root at the left, right child on top."""
    if node is None:
        return
    print_tree(node.right, indent + 1)
    print("     " * indent + str(node.val))
    print_tree(node.left, indent + 1)


# The tree for 2 * 3 + 4:
expr = Node("+", Node("*", Node(2), Node(3)), Node(4))
print_tree(expr)

### Exercise 1.1 — Evaluate an Expression Tree

Write `evaluate(node)` that computes the value of an expression tree.

**Before you code:**
- If the node's value is a number (leaf) — what is the answer?
- If it is `"+"` — what two smaller problems must you solve first?
  (This is recursion on a tree: each child is a smaller tree.)

Support `+`, `-`, `*`, `/`.

```python
evaluate(expr)   # 10   (2*3 + 4)
```

In [ ]:
def evaluate(node):
    pass  # replace this


assert evaluate(expr) == 10
assert evaluate(Node(42)) == 42
assert evaluate(Node("-", Node(10), Node(4))) == 6
print("evaluate: OK")

### Exercise 1.2 — Build a Tree Yourself

Build the tree for `(2 + 3) * (10 - 4)` using `Node`, print it with
`print_tree`, and check that `evaluate` returns `30`.

Notice: the tree needs **no parentheses**. The structure *is* the precedence.

In [ ]:
expr2 = None  # replace this: build the tree for (2 + 3) * (10 - 4)


print_tree(expr2)
assert evaluate(expr2) == 30
print("expr2: OK")

---
## Part 2 — The Binary Search Tree

Back to our fast-lookup problem. Here is the one rule that makes a tree searchable:

> **BST rule:** everything in a node's **left** subtree is *smaller* than the
> node's value; everything in the **right** subtree is *larger*.

Play with the animation before coding: <https://www.cs.usfca.edu/~galles/visualization/BST.html>
Insert 10, 12, 1, 11, 13, 3 and watch where each lands.

### Exercise 2.1 — Insert

Write `insert(node, val)` that inserts `val` into the tree rooted at `node`
and **returns the root** of the resulting tree.

**Think recursively:**
- If `node` is `None` — where does the new value go?
- If `val < node.val` — which subtree does it belong to?
  Insert it there and reattach: `node.left = insert(node.left, val)`.

```python
root = None
for v in [10, 12, 1, 11, 13, 3]:
    root = insert(root, v)
print_tree(root)
```

Compare the printed tree with what the animation showed you.

In [ ]:
def insert(node, val):
    pass  # replace this: return the (possibly new) root of this subtree


root = None
for v in [10, 12, 1, 11, 13, 3]:
    root = insert(root, v)

print_tree(root)
assert root.val == 10
assert root.left.val == 1
assert root.right.val == 12
assert root.left.right.val == 3
assert root.right.left.val == 11
assert root.right.right.val == 13
print("insert: OK")

### Exercise 2.2 — Find

Write `contains(node, val)` that returns `True` if `val` is in the tree.

**The payoff:** at each node you compare once and discard *half* the tree —
the same halving you invented in the Binary Search chapter.
That is why find is O(log n)... *if* the tree is balanced (more on that soon).

```python
contains(root, 11)   # True
contains(root, 7)    # False
```

In [ ]:
def contains(node, val):
    pass  # replace this


assert contains(root, 11) == True
assert contains(root, 10) == True
assert contains(root, 13) == True
assert contains(root, 7)  == False
assert contains(None, 5)  == False
print("contains: OK")

### Exercise 2.3 — Walk the Tree in Order

Write `in_order(node)` that returns a list of all values using this recipe:
first everything in the left subtree, then the node itself, then the right subtree.

```python
in_order(root)   # ?
```

Run it. **Look carefully at the output.** What did you just get for free?
(You have invented *tree sort*: insert n values, walk once, sorted output.)

In [ ]:
def in_order(node):
    pass  # replace this: return a list


result = in_order(root)
print(result)
assert result == [1, 3, 10, 11, 12, 13]

# tree sort on random data:
values = random.sample(range(1000), 50)
r = None
for v in values:
    r = insert(r, v)
assert in_order(r) == sorted(values)
print("in_order: OK — the BST keeps data sorted at all times")

### Exercise 2.4 — Find the Minimum

Where does the smallest value in a BST live? Trace it on the picture.

Write `find_min(node)` that returns the smallest value in a non-empty tree.
No comparisons needed — just walking.

```python
find_min(root)   # 1
```

In [ ]:
def find_min(node):
    pass  # replace this


assert find_min(root) == 1
assert find_min(root.right) == 11
assert find_min(Node(42)) == 42
print("find_min: OK")

### Exercise 2.5 — Delete

Deletion is the tricky one. Watch it in the animation first
(<https://www.cs.usfca.edu/~galles/visualization/BST.html> — insert some values, then delete).

Write `delete(node, val)` that removes `val` and returns the root of the
resulting subtree. Three cases when you find the node:

1. **Leaf** (no children) — just remove it. What do you return?
2. **One child** — the child takes its place.
3. **Two children** — you cannot just remove it; a hole would split the tree.
   Trick: replace the node's value with its **successor** — the *smallest value
   in its right subtree* (you just wrote `find_min`!) — then delete that
   successor from the right subtree. Why is the successor guaranteed to have
   at most one child?

```python
root = delete(root, 1)     # leaf
root = delete(root, 12)    # two children
in_order(root)             # [3, 10, 11, 13]
```

In [ ]:
def delete(node, val):
    pass  # replace this: return the root of this subtree after deletion


# rebuild a fresh tree
root = None
for v in [10, 12, 1, 11, 13, 3]:
    root = insert(root, v)

root = delete(root, 1)              # case: node with one child (1 has right child 3)
assert in_order(root) == [3, 10, 11, 12, 13]

root = delete(root, 13)             # case: leaf
assert in_order(root) == [3, 10, 11, 12]

root = delete(root, 10)             # case: two children (the root itself!)
assert in_order(root) == [3, 11, 12]

root = delete(root, 99)             # deleting a missing value changes nothing
assert in_order(root) == [3, 11, 12]
print("delete: OK")

# stress test: delete everything in random order
values = random.sample(range(500), 100)
r = None
for v in values:
    r = insert(r, v)
random.shuffle(values)
remaining = sorted(values)
for v in values:
    r = delete(r, v)
    remaining.remove(v)
    assert in_order(r) == remaining, f"tree broken after deleting {v}"
print("delete stress test: OK")

### The Catch — Balance

Insert `1, 2, 3, 4, 5, 6, 7` in that order and print the tree. What shape do you get?

Every insert went right: the "tree" is a chain — a glorified linked list, and
find is back to O(n). Sorted input is the *worst* input for a plain BST.

Self-balancing trees (AVL, Red-Black) fix this by rotating nodes on insert to keep
the height at log n. Watch one rebalance itself:
<https://www.cs.usfca.edu/~galles/visualization/RedBlack.html> — insert 1..10 in order
and watch it stay shallow. (Databases use B-Trees — the same idea, made disk-friendly.)

In [ ]:
chain = None
for v in [1, 2, 3, 4, 5, 6, 7]:
    chain = insert(chain, v)
print_tree(chain)

---
## Part 3 — The Scheduler Problem

You are building a task scheduler (like Unix cron). Tasks arrive at any time:

- task1: run at 1:00pm
- task2: run at 12:30pm
- task3: run at 2:00pm

The scheduler repeatedly needs just **one** thing: *the earliest task*.
So the operations are: `add(task)`, `pop_earliest()`. Nothing else. No search,
no arbitrary delete.

**Before reading on**, rate each strategy (fill the ?s in your head):

| Strategy | add | pop earliest |
|----------|-----|--------------|
| unsorted list | O(1) | O(?) scan |
| keep list sorted | O(?) | O(1) |
| BST | O(log n) | O(log n) |
| one slot per second for 1000 days | O(1) | O(?) — and how much memory? |

The BST wins — but it maintains *total* order, and we only ever ask for the min.
Can a structure that promises **less** do the job with less work?
It exists: the **min-heap**.

### The Heap Shape

A min-heap is a binary tree with one weak promise:

> **Heap rule:** every node is ≤ its children. (Nothing about left vs right!)

So the minimum is always... where?

Because the shape is always a "filled top-down, left-to-right" tree, we don't
need Node objects at all — we can store it in a **plain list**:

```
        1
      /   \
     3     2          stored as:  [1, 3, 2, 8, 5]
    / \
   8   5
```

### Exercise 3.1 — The Index Arithmetic

For the node at list index `i`, figure out (try it on the picture above):

- `parent(i)` = ?
- `left_child(i)` = ?
- `right_child(i)` = ?

Write the three functions.

```python
parent(3)       # 1   (8's parent is 3)
left_child(1)   # 3
right_child(0)  # 2
```

In [ ]:
def parent(i):
    pass  # replace this

def left_child(i):
    pass  # replace this

def right_child(i):
    pass  # replace this


assert parent(3) == 1 and parent(4) == 1 and parent(1) == 0 and parent(2) == 0
assert left_child(0) == 1 and right_child(0) == 2
assert left_child(1) == 3 and right_child(1) == 4
print("index arithmetic: OK")

### Exercise 3.2 — Push (Bubble Up)

To add a value: append it at the end of the list (keeping the shape), then
repair the heap rule — while the new value is *smaller than its parent*,
swap them, and keep climbing.

Write `heap_push(heap, val)` (in place).

```python
h = [1, 3, 2, 8, 5]
heap_push(h, 0)
h   # [0, 3, 1, 8, 5, 2]  — 0 climbed to the top
```

How many swaps can a push cost at most, for a heap of n items?
(The tree has log₂(n) levels — that is the whole point.)

In [ ]:
def heap_push(heap, val):
    pass  # replace this


h = [1, 3, 2, 8, 5]
heap_push(h, 0)
assert h == [0, 3, 1, 8, 5, 2]

h = []
for v in [5, 3, 8, 1]:
    heap_push(h, v)
assert h[0] == 1, "minimum must be at the root"
print("heap_push: OK")

### Exercise 3.3 — Pop the Minimum (Sink Down)

The minimum is `heap[0]`. Removing it leaves a hole at the root. To keep the
shape, move the **last** element into the hole — then repair: while it is
*bigger than one of its children*, swap it with its **smaller child** and keep sinking.

Write `heap_pop(heap)` that removes and returns the minimum. In place.

**Edge cases:** why must you swap with the *smaller* child, not just any child?
What if a node has only a left child?

```python
h = [1, 3, 2, 8, 5]
heap_pop(h)   # 1
heap_pop(h)   # 2
heap_pop(h)   # 3
```

In [ ]:
def heap_pop(heap):
    pass  # replace this


h = [1, 3, 2, 8, 5]
assert heap_pop(h) == 1
assert heap_pop(h) == 2
assert heap_pop(h) == 3
assert heap_pop(h) == 5
assert heap_pop(h) == 8
assert h == []

# stress: pushes and pops always yield values in sorted order
values = [random.randint(0, 1000) for _ in range(500)]
h = []
for v in values:
    heap_push(h, v)
popped = [heap_pop(h) for _ in range(len(values))]
assert popped == sorted(values)
print("heap_pop: OK")

### Exercise 3.4 — The Scheduler, Solved

Your heap works on anything comparable — including tuples, which Python
compares by first element. So a task is just `(time, name)`.

Write `run_scheduler(tasks)` that takes a list of `(time, name)` tuples,
pushes them all into a heap, then pops until empty, returning the task names
in execution order.

```python
run_scheduler([(1300, "task1"), (1230, "task2"), (1400, "task3"), (1100, "task4")])
# ["task4", "task2", "task1", "task3"]
```

In [ ]:
def run_scheduler(tasks):
    pass  # replace this


order = run_scheduler([(1300, "task1"), (1230, "task2"), (1400, "task3"), (1100, "task4")])
assert order == ["task4", "task2", "task1", "task3"]
print("run_scheduler: OK")

### Exercise 3.5 — Heap Sort, and the Standard Library

1. You already proved it in the stress test: push n values, pop n values → sorted.
   Package it as `heap_sort(lst)` returning a new sorted list.
   Each of the 2n operations costs O(log n) — so heap sort is **O(n log n)**,
   in place-able, no merging needed.

2. Python ships your invention as the `heapq` module: `heapq.heappush`,
   `heapq.heappop`, and `heapq.heapify` (which builds a heap from a list in O(n)).
   Verify your heap agrees with it.

```python
heap_sort([5, 2, 9, 1])   # [1, 2, 5, 9]
```

In [ ]:
import heapq

def heap_sort(lst):
    pass  # replace this: use your heap_push / heap_pop


assert heap_sort([5, 2, 9, 1]) == [1, 2, 5, 9]
assert heap_sort([]) == []

# agree with the standard library on random data
data = [random.randint(0, 100) for _ in range(200)]
mine = list(data)
h = []
for v in mine:
    heap_push(h, v)

theirs = list(data)
heapq.heapify(theirs)

assert heap_pop(h) == heapq.heappop(theirs)
assert heap_pop(h) == heapq.heappop(theirs)
assert heap_pop(h) == heapq.heappop(theirs)
print("heap_sort + heapq agreement: OK")

---
## Summary

| Structure | You built | Sweet spot |
|-----------|-----------|-----------|
| Expression tree | `evaluate` — recursion on trees | how compilers see code |
| Binary search tree | `insert`, `contains`, `in_order`, `find_min`, `delete` | ordered data with frequent insert/delete, all O(log n) *if balanced* |
| Min-heap | `heap_push`, `heap_pop`, `run_scheduler`, `heap_sort` | only need the min: schedulers, priority queues, O(log n) with a plain list |

The trade-off ladder you climbed:
- A **dictionary** promises the most (any-key O(1)) and pays in memory and lost ordering.
- A **BST** promises total order and pays log n per operation — plus a balance headache
  (solved by AVL / Red-Black trees; B-Trees take the same idea to disk in every database).
- A **heap** promises the least — just the minimum — and is the simplest and fastest of all.

**Choosing a data structure = paying only for the promises you actually use.**